In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from tqdm import tqdm

### 模块一：市场共振风险（最核心！Alpha失效第一信号）
系统性大跌最典型的特征，不是跌得多，是**全市场走势趋同、个股失去分化**。

当市场进入共振状态，选股Alpha彻底失效，无论怎么分散、怎么精选个股，都会跟随大盘集体下跌。

#### 核心监控指标
1. **全市场个股平均相关系数**
下跌共振期，个股两两相关性会飙升至历史高位，是多头最大杀手。单独区分**上行相关性、下行相关性**，只把恐慌性同向下跌纳入风险统计。
2. **市场结构扩散度**
上涨家数占比、创新高/新低比值、20日均线上个股占比，快速捕捉市场赚钱效应崩塌信号。
3. **横截面收益离散度**
离散度走低=个股走势统一，系统风险主导市场；离散度走高=个股分化，选股策略友好。

#### 风控结论
**相关性飙升+离散度下行=系统性风险确认，优先降Beta，其次降仓位。**

In [ ]:
# ====================== 配置参数 ======================
INDEX_CODE = "000985.XSHG"
LOOKBACK_DAYS = 100        # 拉取K线数量
MA_WINDOW = 20             # 20日均线（扩散度）
CORR_WINDOW = 60           # 相关系数滚动窗口
SAMPLE_STOCKS = 4881        # 抽样个股数量，400推荐；300更快，500精度更高
MIN_VALID_RATIO = 0.8      # 个股有效数据最低占比，过滤烂数据
date_today = datetime.now()

In [ ]:
# 1. 获取中证全指全部成分股
stocks_all = get_index_stocks(INDEX_CODE, date=date_today)
print(f"中证全指原始成分股数量：{len(stocks_all)}")

In [ ]:
# 2. 一次性拉全部成分股收盘价
df_price = get_price(
    stocks_all,
    end_date=date_today,
    count=LOOKBACK_DAYS,
    frequency='1d',
    fields=['close'],
    skip_paused=False,
    fq='pre',
    panel=False,
    fill_paused=True
)
print(f"df_price shape: {df_price.shape}")
df_close = df_price.pivot(index="time", columns="code", values="close")
df_ret = df_close.pct_change()
df_ret = df_ret.dropna(how="all")
print(f"收益率矩阵 shape (日期 × 个股): {df_ret.shape}")

In [ ]:
# ========== 新增：清洗个股，剔除缺失过多标的 ==========
valid_ratio = df_ret.count(axis=0) / len(df_ret)
good_stocks = valid_ratio[valid_ratio >= MIN_VALID_RATIO].index.tolist()
print(f"清洗后，有效股票数量：{len(good_stocks)}")

In [ ]:
# ====================== 指标1：横截面收益离散度 ======================
# 横截面离散度，要用全部有效个股，不要抽样，保证指标准确性
cross_section_std = df_ret[good_stocks].std(axis=1)
cross_section_std.name = "横截面离散度"

In [ ]:
cross_section_std.plot(figsize=(10,5))

In [ ]:
# ====================== 指标2：市场扩散度：20日均线上个股占比 ======================
df_close_good = df_close[good_stocks]
df_ma20 = df_close_good.rolling(window=MA_WINDOW).mean()
above_ma20 = (df_close_good > df_ma20)
diffusion_ratio = above_ma20.sum(axis=1) / above_ma20.count(axis=1)
diffusion_ratio.name = "20日均线以上个股占比_扩散度"

In [ ]:
diffusion_ratio.plot(figsize=(10,5))

In [ ]:
# ====================== 指标3 滚动平均相关系数 ======================


# ============================================================
# 1. 高速计算：所有股票两两 Pearson correlation 的平均值
# ============================================================
def _avg_corr_fast(ret_arr: np.ndarray) -> float:
    """
    高速计算：

        mean_{i<j} corr(X_i, X_j)

    不构造 N×N correlation matrix。

    数学恒等式：

        corr(i,j)
        = 1/(T-1) * sum_t(Z_ti * Z_tj)

    而：

        sum_{i<j} Z_ti Z_tj
        = [ (sum_i Z_ti)^2 - sum_i Z_ti^2 ] / 2

    标准化以后：

        sum_t Z_ti^2 = T - 1

    因此只需要计算每个交易日所有股票标准化收益率的横截面总和。

    ------------------------------------------------------------
    参数
    ------------------------------------------------------------
    ret_arr : np.ndarray
        shape = (T, N)
        T = 交易日数量
        N = 股票数量

    ------------------------------------------------------------
    注意
    ------------------------------------------------------------
    本函数假设 ret_arr 中不存在 NaN / inf。

    如果存在 NaN，请使用下面提供的 NaN 版本，或者先清洗数据。
    """

    T, N = ret_arr.shape

    # 至少需要 2 个观测、2 只股票
    if T < 2 or N < 2:
        return np.nan

    # --------------------------------------------------------
    # 检查 finite
    # --------------------------------------------------------
    if not np.isfinite(ret_arr).all():
        return np.nan

    # --------------------------------------------------------
    # 均值和样本标准差
    #
    # ddof=1 与 pandas corr() 使用的 Pearson correlation 对齐
    # --------------------------------------------------------
    mean = ret_arr.mean(axis=0)
    std = ret_arr.std(axis=0, ddof=1)

    # --------------------------------------------------------
    # 去掉常数列
    #
    # pandas corr() 对常数列产生 NaN。
    # 这里不能因为一列是常数，就让整个结果变成 NaN。
    # --------------------------------------------------------
    valid = std > 1e-12

    n_valid = int(valid.sum())

    if n_valid < 2:
        return np.nan

    # --------------------------------------------------------
    # 只保留有波动的股票
    # --------------------------------------------------------
    x = ret_arr[:, valid]
    mean_valid = mean[valid]
    std_valid = std[valid]

    # --------------------------------------------------------
    # 每只股票标准化后的缩放因子
    # --------------------------------------------------------
    inv_std = 1.0 / std_valid

    # --------------------------------------------------------
    # 对于：
    #
    # Z = (X - mean) / std
    #
    # 每一天：
    #
    # sum_i Z_ti
    #
    # = sum_i X_ti / std_i
    #   - sum_i mean_i / std_i
    #
    # 所以不需要显式创建 Z
    # --------------------------------------------------------
    c = float(mean_valid @ inv_std)

    row_sums = x @ inv_std - c

    # --------------------------------------------------------
    # sum_t (sum_i Z_ti)^2
    # --------------------------------------------------------
    sum_sq_rows = float(row_sums @ row_sums)

    # --------------------------------------------------------
    # 标准化以后：
    #
    # sum_t Z_ti² = T - 1
    #
    # 一共有 n_valid 列
    # --------------------------------------------------------
    sum_sq_cols = (T - 1) * n_valid

    # --------------------------------------------------------
    # 所有 i < j 的 correlation 之和
    #
    # sum_{i<j} corr_ij
    # =
    # [sum_t(sum_i Z_ti)^2 - sum_i sum_t Z_ti²]
    # / [2(T-1)]
    # --------------------------------------------------------
    off_diag_corr_sum = (
        sum_sq_rows - sum_sq_cols
    ) / (2.0 * (T - 1))

    # --------------------------------------------------------
    # 股票两两组合数量
    # --------------------------------------------------------
    n_pairs = n_valid * (n_valid - 1) / 2.0

    # --------------------------------------------------------
    # 平均 correlation
    # --------------------------------------------------------
    avg_corr = off_diag_corr_sum / n_pairs

    return float(avg_corr)


# ============================================================
# 2. 单个窗口计算
# ============================================================
def calc_avg_corr_fast(
    ret_arr: np.ndarray,
    min_down_days: int = 20,
    min_up_days: int = 20
):
    """
    计算单个滚动窗口：

        1. 全部交易日平均相关系数
        2. 下行市场平均相关系数
        3. 上行市场平均相关系数

    参数
    ----
    ret_arr : np.ndarray
        shape = (T, N)

    min_down_days : int
        下行交易日最少数量

    min_up_days : int
        上行交易日最少数量

    返回
    ----
    avg_all
    avg_down
    avg_up
    """

    # --------------------------------------------------------
    # 市场收益率
    #
    # 与原代码：
    #
    # market_ret = ret_df.mean(axis=1)
    #
    # 对没有 NaN 的数据完全一致
    # --------------------------------------------------------
    market_ret = ret_arr.mean(axis=1)

    # --------------------------------------------------------
    # 上下行市场
    # --------------------------------------------------------
    mask_down = market_ret < 0
    mask_up = market_ret > 0

    n_down = int(mask_down.sum())
    n_up = int(mask_up.sum())

    # --------------------------------------------------------
    # 全部交易日
    # --------------------------------------------------------
    avg_all = _avg_corr_fast(ret_arr)

    # --------------------------------------------------------
    # 下行
    # --------------------------------------------------------
    if n_down >= min_down_days:
        avg_down = _avg_corr_fast(ret_arr[mask_down])
    else:
        avg_down = np.nan

    # --------------------------------------------------------
    # 上行
    # --------------------------------------------------------
    if n_up >= min_up_days:
        avg_up = _avg_corr_fast(ret_arr[mask_up])
    else:
        avg_up = np.nan

    return avg_all, avg_down, avg_up


# ============================================================
# 3. 滚动计算主函数
# ============================================================
def rolling_avg_corr_fast(
    df_ret_sample: pd.DataFrame,
    corr_window: int,
    min_down_days: int = 20,
    min_up_days: int = 20,
    show_progress: bool = True
):
    """
    高速滚动平均相关系数。

    返回：
        result_df

    columns:
        avg_corr_all
        avg_corr_down
        avg_corr_up
    """

    # ========================================================
    # 基本检查
    # ========================================================
    if not isinstance(df_ret_sample, pd.DataFrame):
        raise TypeError("df_ret_sample 必须是 pandas DataFrame")

    if corr_window < 2:
        raise ValueError("corr_window 必须 >= 2")

    if len(df_ret_sample) <= corr_window:
        raise ValueError(
            f"数据长度 {len(df_ret_sample)} "
            f"不足以计算 corr_window={corr_window}"
        )

    # ========================================================
    # 一次性转 NumPy
    #
    # 后面所有 rolling window 都直接从这个数组切 view
    # ========================================================
    ret_array = df_ret_sample.to_numpy(
        dtype=np.float64,
        copy=False
    )

    # ========================================================
    # 检查 NaN / inf
    # ========================================================
    if not np.isfinite(ret_array).all():

        nan_count = np.isnan(ret_array).sum()
        inf_count = np.isinf(ret_array).sum()

        raise ValueError(
            "df_ret_sample 中存在 NaN / inf。\n"
            f"NaN 数量: {nan_count:,}\n"
            f"Inf 数量: {inf_count:,}\n\n"
            "当前高速 O(TN) 算法要求输入窗口没有 NaN / inf。\n"
            "如果你的数据确实有 NaN，需要使用 pairwise NaN 版本。"
        )

    # ========================================================
    # 日期
    # ========================================================
    date_index = df_ret_sample.index

    # ========================================================
    # 股票数量
    # ========================================================
    n_days, n_stocks = ret_array.shape

    print(
        f"数据维度: {n_days:,} 个交易日 × "
        f"{n_stocks:,} 只股票"
    )

    print(
        f"滚动窗口: {corr_window:,} 天"
    )

    # ========================================================
    # 输出数组
    #
    # 比 list append 更适合大量滚动计算
    # ========================================================
    n_results = n_days - corr_window

    rolling_all = np.full(
        n_results,
        np.nan,
        dtype=np.float64
    )

    rolling_down = np.full(
        n_results,
        np.nan,
        dtype=np.float64
    )

    rolling_up = np.full(
        n_results,
        np.nan,
        dtype=np.float64
    )

    # ========================================================
    # 滚动计算
    # ========================================================
    iterator = range(
        corr_window,
        n_days
    )

    if show_progress:
        iterator = tqdm(
            iterator,
            total=n_results,
            desc="计算滚动平均相关系数"
        )

    for result_idx, i in enumerate(iterator):

        # ----------------------------------------------------
        # NumPy slice
        #
        # 这里是 view，不是 copy
        # ----------------------------------------------------
        sub_ret = ret_array[
            i - corr_window:i
        ]

        try:

            # ------------------------------------------------
            # 市场收益率
            # ------------------------------------------------
            market_ret = sub_ret.mean(axis=1)

            mask_down = market_ret < 0
            mask_up = market_ret > 0

            n_down = int(mask_down.sum())
            n_up = int(mask_up.sum())

            # ------------------------------------------------
            # 全部
            # ------------------------------------------------
            rolling_all[result_idx] = _avg_corr_fast(
                sub_ret
            )

            # ------------------------------------------------
            # 下行
            # ------------------------------------------------
            if n_down >= min_down_days:

                rolling_down[result_idx] = _avg_corr_fast(
                    sub_ret[mask_down]
                )

            # ------------------------------------------------
            # 上行
            # ------------------------------------------------
            if n_up >= min_up_days:

                rolling_up[result_idx] = _avg_corr_fast(
                    sub_ret[mask_up]
                )

        except Exception as e:

            print(
                f"\n窗口 {i} 计算失败: "
                f"{str(e)[:200]}"
            )

            rolling_all[result_idx] = np.nan
            rolling_down[result_idx] = np.nan
            rolling_up[result_idx] = np.nan

    # ========================================================
    # 结果日期
    #
    # 与你的原代码完全一致：
    #
    # date_list.append(df_ret_sample.index[i])
    #
    # 即：
    # 第一个结果对应第 corr_window 个交易日
    # ========================================================
    result_index = date_index[corr_window:]

    # ========================================================
    # DataFrame
    # ========================================================
    result_df = pd.DataFrame(
        {
            "avg_corr_all": rolling_all,
            "avg_corr_down": rolling_down,
            "avg_corr_up": rolling_up,
        },
        index=result_index
    )

    return result_df

In [ ]:
df_corr = rolling_avg_corr_fast(
    df_ret_sample=df_ret,
    corr_window=CORR_WINDOW,
    min_down_days=20,
    min_up_days=20
)

In [ ]:
df_corr.plot(figsize=(10,5))

In [ ]:
def calc_avg_corr_original(ret_df: pd.DataFrame):
    market_ret = ret_df.mean(axis=1)

    mask_down = market_ret < 0
    mask_up = market_ret > 0

    # 全部
    corr_all = ret_df.corr()

    tri = np.triu(
        np.ones_like(corr_all, dtype=bool),
        k=1
    )

    avg_all = corr_all.where(tri).stack().mean()

    # 下行
    ret_down = ret_df.loc[mask_down]

    if len(ret_down) >= 20:

        corr_down = ret_down.corr()

        tri_d = np.triu(
            np.ones_like(corr_down, dtype=bool),
            k=1
        )

        avg_down = corr_down.where(tri_d).stack().mean()

    else:
        avg_down = np.nan

    # 上行
    ret_up = ret_df.loc[mask_up]

    if len(ret_up) >= 20:

        corr_up = ret_up.corr()

        tri_u = np.triu(
            np.ones_like(corr_up, dtype=bool),
            k=1
        )

        avg_up = corr_up.where(tri_u).stack().mean()

    else:
        avg_up = np.nan

    return avg_all, avg_down, avg_up

In [ ]:
# ============================================================
# 随机测试
# ============================================================

np.random.seed(42)

test_data = pd.DataFrame(
    np.random.randn(250, 100) * 0.02
)

v1 = calc_avg_corr_original(test_data)

v2 = calc_avg_corr_fast(
    test_data.to_numpy(dtype=np.float64)
)

print("第一版:")
print(v1)

print("\n第二版:")
print(v2)

print("\n差异:")
print(
    np.array(v1) - np.array(v2)
)

In [ ]:
df_corr = df_corr.rename(columns={
    'avg_corr_all': '滚动60日_全区间平均相关系数',
    'avg_corr_down': '滚动60日_下行平均相关系数',
    'avg_corr_up':'滚动60日_上行平均相关系数',
})

In [ ]:
# ====================== 合并模块一所有指标 ======================
df_module1 = pd.concat([
    cross_section_std,
    diffusion_ratio,
], axis=1)
df_module1 = df_module1.join(df_corr, how="outer")

In [ ]:
df_module1.head()

In [ ]:
# 输出最新一行
latest = df_module1.iloc[-1]
print("\n===== 模块一【市场共振风险】最新指标 =====")
print(latest)

# 保存结果
df_module1.to_csv("module1_market_resonance_4881_stock.csv", encoding="utf-8-sig")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


# ===================== 绘图 =====================
fig, axes = plt.subplots(3,1, figsize=(16,12), sharex=True)
fig.suptitle("模块一｜市场共振风险指标时序图", fontsize=16, y=0.96)

# 子图1：横截面离散度
ax1 = axes[0]
ax1.plot(df_module1.index, df_module1["横截面离散度"], color="#1f77b4", linewidth=1.5, label="横截面离散度")
ax1.set_title("横截面收益离散度（越低=市场趋同，共振风险上升）")
ax1.legend(loc="upper right")
ax1.set_ylabel("离散度")

# 子图2：市场扩散度，20日均线上个股占比
ax2 = axes[1]
ax2.plot(df_module1.index, df_module1["20日均线以上个股占比_扩散度"], color="#ff7f0e", linewidth=1.5, label="20日均线上个股占比")
ax2.set_title("市场扩散度（越低赚钱效应越弱）")
ax2.legend(loc="upper right")
ax2.set_ylabel("占比")

# 子图3：三组相关系数，重点看【下行平均相关系数】
ax3 = axes[2]
ax3.plot(df_module1.index, df_module1["滚动60日_全区间平均相关系数"], color="#2ca02c", alpha=0.7, label="全区间平均相关")
ax3.plot(df_module1.index, df_module1["滚动60日_上行平均相关系数"], color="#1f77b4", alpha=0.7, label="上行相关")
ax3.plot(df_module1.index, df_module1["滚动60日_下行平均相关系数"], color="#d62728", linewidth=2, label="下行相关【核心预警】")
ax3.set_title("滚动60日个股平均相关系数｜红色=下跌时段共振强度")
ax3.legend(loc="upper right")
ax3.set_ylabel("相关系数")
ax3.set_xlabel("日期")

plt.tight_layout()
plt.show()


In [ ]:
# 计算下行相关系数的历史分位
from scipy import stats

ser_down_corr = df_module1["滚动60日_下行平均相关系数"].dropna()
# 逐行计算分位值
down_corr_pct = ser_down_corr.apply(lambda x: stats.percentileofscore(ser_down_corr, x))

fig, ax = plt.subplots(figsize=(16,5))
ax.plot(ser_down_corr.index, ser_down_corr.values, color="red", lw=1.5, label="下行相关系数")
ax2 = ax.twinx()
ax2.plot(down_corr_pct.index, down_corr_pct.values, color="black", lw=1, linestyle="--", label="历史分位(%)")
ax.set_title("滚动60日下行相关系数 + 历史分位", fontsize=14)
ax.set_ylabel("相关系数", color="red")
ax2.set_ylabel("历史分位 %", color="black")
ax.grid(True, alpha=0.3)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(16,6))
ax.plot(df_module1.index, df_module1["横截面离散度"], label="横截面离散度", lw=1.2)
ax.plot(df_module1.index, df_module1["20日均线以上个股占比_扩散度"], label="20日线上占比", lw=1.2)
ax.plot(df_module1.index, df_module1["滚动60日_下行平均相关系数"], label="下行相关系数", lw=1.5, c='r')
ax.set_title("模块一三大共振指标合并时序", fontsize=14)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()